# Exercice 08 - Ingestion depuis le Web (API)

## Objectifs
- Appeler une API REST avec Python
- Transformer les donnees JSON en DataFrame
- Ingerer les donnees web dans Bronze
- Gerer les erreurs et les retries

---

## 1. Architecture d'ingestion Web

```
+------------------+                    +------------------------+
|                  |                    |        MinIO           |
|    INTERNET      |     PYTHON         |                        |
|                  |    + SPARK         |  +------------------+  |
|  +------------+  |  =============>    |  |     BRONZE       |  |
|  | API REST   |  |                    |  +------------------+  |
|  | (JSON)     |  |                    |  | /api_users/      |  |
|  +------------+  |                    |  |   /2024-01-15/   |  |
|                  |                    |  | /api_posts/      |  |
|  +------------+  |                    |  |   /2024-01-15/   |  |
|  | Fichiers   |  |                    |  +------------------+  |
|  | CSV/JSON   |  |                    |                        |
|  +------------+  |                    +------------------------+
|                  |
+------------------+

Etapes :
1. Appeler l'API avec requests
2. Parser le JSON
3. Creer un DataFrame Spark
4. Sauvegarder dans Bronze
```

## 2. Configuration

In [ ]:
from pyspark.sql import SparkSession
from datetime import datetime
import requests
import json

# Creer la SparkSession
spark = SparkSession.builder \
    .appName("Ingestion Web") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.1,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

date_ingestion = datetime.now().strftime("%Y-%m-%d")
print(f"Spark pret - Date : {date_ingestion}")

## 3. Appeler une API REST simple

In [ ]:
# API de test : JSONPlaceholder (API publique gratuite)
url_users = "https://jsonplaceholder.typicode.com/users"

# Appeler l'API
response = requests.get(url_users)

print(f"Status code : {response.status_code}")
print(f"Content-Type : {response.headers.get('Content-Type')}")

In [ ]:
# Parser le JSON
data = response.json()

print(f"Type : {type(data)}")
print(f"Nombre d'elements : {len(data)}")
print("\nPremier element :")
print(json.dumps(data[0], indent=2))

## 4. Convertir en DataFrame Spark

In [ ]:
# Creer un DataFrame depuis une liste Python
# Les donnees de l'API ont des structures imbriquees (address, company, geo)
# On definit le schema explicitement pour eviter les erreurs d'inference

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Schema pour les donnees JSONPlaceholder /users
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("username", StringType(), True),
    StructField("email", StringType(), True),
    StructField("address", StructType([
        StructField("street", StringType(), True),
        StructField("suite", StringType(), True),
        StructField("city", StringType(), True),
        StructField("zipcode", StringType(), True),
        StructField("geo", StructType([
            StructField("lat", StringType(), True),
            StructField("lng", StringType(), True)
        ]), True)
    ]), True),
    StructField("phone", StringType(), True),
    StructField("website", StringType(), True),
    StructField("company", StructType([
        StructField("name", StringType(), True),
        StructField("catchPhrase", StringType(), True),
        StructField("bs", StringType(), True)
    ]), True)
])

df_users = spark.createDataFrame(data, schema=schema)

print("Schema :")
df_users.printSchema()

In [ ]:
# Afficher les donnees
df_users.select("id", "name", "email", "phone").show()

In [ ]:
# Aplatir les structures imbriquees
from pyspark.sql import functions as F

df_users_flat = df_users.select(
    "id",
    "name",
    "username",
    "email",
    "phone",
    "website",
    F.col("address.street").alias("street"),
    F.col("address.city").alias("city"),
    F.col("address.zipcode").alias("zipcode"),
    F.col("company.name").alias("company_name")
)

df_users_flat.show(truncate=False)

## 5. Sauvegarder dans Bronze

In [ ]:
# Ajouter les metadonnees
df_final = df_users_flat \
    .withColumn("_source", F.lit("api_jsonplaceholder")) \
    .withColumn("_endpoint", F.lit("users")) \
    .withColumn("_ingestion_date", F.lit(date_ingestion))

# Sauvegarder
chemin_bronze = f"s3a://bronze/api_users/{date_ingestion}"
df_final.write.mode("overwrite").parquet(chemin_bronze)

print(f"Sauvegarde reussie : {chemin_bronze}")
print(f"Lignes ingerees : {df_final.count()}")

## 6. Fonction d'ingestion API generique

In [ ]:
def ingerer_api(url, nom_dataset, date=None):
    """
    Ingere des donnees depuis une API REST vers Bronze.
    
    Args:
        url: URL de l'API
        nom_dataset: Nom du dataset pour le chemin Bronze
        date: Date d'ingestion (optionnel)
    
    Returns:
        dict: Statistiques d'ingestion
    """
    if date is None:
        date = datetime.now().strftime("%Y-%m-%d")
    
    try:
        # Appeler l'API
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        # Parser le JSON
        data = response.json()
        
        # Creer le DataFrame
        if isinstance(data, list):
            df = spark.createDataFrame(data)
        else:
            df = spark.createDataFrame([data])
        
        # Ajouter les metadonnees
        df = df.withColumn("_source", F.lit(url)) \
               .withColumn("_ingestion_date", F.lit(date))
        
        # Sauvegarder
        chemin = f"s3a://bronze/{nom_dataset}/{date}"
        df.write.mode("overwrite").parquet(chemin)
        
        nb_lignes = df.count()
        print(f"[OK] {nom_dataset} : {nb_lignes} lignes -> {chemin}")
        
        return {"status": "OK", "lignes": nb_lignes, "chemin": chemin}
        
    except requests.exceptions.RequestException as e:
        print(f"[ERREUR] {nom_dataset} : Erreur API - {e}")
        return {"status": "ERREUR", "erreur": str(e)}
    except Exception as e:
        print(f"[ERREUR] {nom_dataset} : {e}")
        return {"status": "ERREUR", "erreur": str(e)}

In [ ]:
# Tester avec d'autres endpoints
endpoints = [
    ("https://jsonplaceholder.typicode.com/posts", "api_posts"),
    ("https://jsonplaceholder.typicode.com/comments", "api_comments"),
    ("https://jsonplaceholder.typicode.com/todos", "api_todos")
]

print("Ingestion multi-endpoints :")
print("=" * 50)

for url, nom in endpoints:
    ingerer_api(url, nom, date_ingestion)

## 7. Gestion des erreurs et retries

In [ ]:
import time

def appeler_api_avec_retry(url, max_retries=3, delay=2):
    """
    Appelle une API avec retry en cas d'erreur.
    
    Args:
        url: URL de l'API
        max_retries: Nombre maximum de tentatives
        delay: Delai entre les tentatives (secondes)
    
    Returns:
        Donnees JSON ou None
    """
    for tentative in range(max_retries):
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            return response.json()
            
        except requests.exceptions.RequestException as e:
            print(f"  Tentative {tentative + 1}/{max_retries} echouee : {e}")
            if tentative < max_retries - 1:
                print(f"  Attente de {delay} secondes...")
                time.sleep(delay)
                delay *= 2  # Backoff exponentiel
    
    print(f"  Echec apres {max_retries} tentatives")
    return None

In [ ]:
# Test avec une URL valide
data = appeler_api_avec_retry("https://jsonplaceholder.typicode.com/albums")
if data:
    print(f"Succes : {len(data)} elements recuperes")

## 8. Ingerer un fichier CSV depuis le Web

In [ ]:
# Telecharger un CSV depuis une URL
url_csv = "https://raw.githubusercontent.com/datasets/covid-19/main/data/countries-aggregated.csv"

# Telecharger et lire le CSV
response = requests.get(url_csv)
if response.status_code == 200:
    # Sauvegarder temporairement
    with open("/tmp/covid_data.csv", "w") as f:
        f.write(response.text)
    
    # Lire avec Spark
    df_covid = spark.read.csv("/tmp/covid_data.csv", header=True, inferSchema=True)
    print(f"Donnees COVID : {df_covid.count()} lignes")
    df_covid.show(5)

In [ ]:
# Sauvegarder dans Bronze
chemin = f"s3a://bronze/covid_data/{date_ingestion}"
df_covid.write.mode("overwrite").parquet(chemin)
print(f"Sauvegarde : {chemin}")

---

## Exercice

**Objectif** : Ingerer des donnees depuis une API publique

**Consigne** :
1. Utilisez l'API REST Countries : `https://restcountries.com/v3.1/all`
2. Extrayez les champs : nom, capitale, region, population, superficie
3. Sauvegardez dans Bronze

A vous de jouer :

In [ ]:
# TODO: Appeler l'API REST Countries
url_countries = "https://restcountries.com/v3.1/all"

In [ ]:
# TODO: Extraire les champs utiles et creer un DataFrame

In [ ]:
# TODO: Sauvegarder dans Bronze

---

## Resume

Dans ce notebook, vous avez appris :
- Comment **appeler une API REST** avec Python requests
- Comment **convertir du JSON** en DataFrame Spark
- Comment **aplatir** les structures imbriquees
- Comment **gerer les erreurs** avec retry
- Comment **ingerer des fichiers** CSV depuis le Web

### Prochaine etape
Dans le prochain notebook, nous apprendrons a nettoyer les donnees pour la couche Silver.